# Unit 1 Hands-on: Generative AI & NLP Fundamentals

This notebook demonstrates basic concepts of Generative AI and NLP using pre-trained models from Hugging Face.


## 1. Introduction & Setup

In this section, we import required libraries and set up the environment for working with Generative AI models.


### Installing Required Libraries

The following libraries are required:
- transformers
- torch
- nltk


In [7]:
!pip install transformers torch nltk



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


### Loading Course Material

We load the text file containing Unit 1 content which will be used as context for NLP tasks.


In [ ]:
from transformers import pipeline, set_seed, GPT2Tokenizer
import nltk
import os


In [ ]:
file_path = "unit 1.txt"

In [ ]:
try:
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()
    print("File loaded successfully!")
except FileNotFoundError:
    print(f"Error: '{file_path}' not found.")

### Preview of Loaded Data

The first few characters of the text are displayed to verify successful loading.


In [ ]:
print("--- Data Preview ---")
print(text[:500] + "...")

## 2. Generative AI: Dumb vs Smart Models

We compare a smaller model (distilgpt2) with a standard model (gpt2) to observe differences in text generation quality.


### Setting Random Seed

A seed ensures reproducibility of generated text.


In [ ]:
set_seed(42)

### Defining Prompt

Both models will generate text starting from the same prompt.


In [ ]:
prompt = "Generative AI is a revolutionary technology that"

### Fast Model: distilgpt2

This model is smaller and faster but may generate less coherent text.


In [ ]:
# Initialize the pipeline with the specific model
fast_generator = pipeline('text-generation', model='distilgpt2')

# Generate text
output_fast = fast_generator(prompt, max_length=50, num_return_sequences=1)
print(output_fast[0]['generated_text'])

### Smart Model: gpt2

This model produces more coherent and context-aware text.


In [ ]:
smart_generator = pipeline('text-generation', model='gpt2')

output_smart = smart_generator(prompt, max_length=50, num_return_sequences=1)
print(output_smart[0]['generated_text'])

### Observation

The gpt2 model generates more meaningful and coherent responses compared to distilgpt2, which sometimes repeats phrases or drifts from the topic.


In [ ]:
# 1. Initialize the Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

In [ ]:
tokens = tokenizer.tokenize(sample_sentence)
print(f"Tokens: {tokens}")

In [ ]:
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(f"Token IDs: {token_ids}")

In [ ]:
# Download necessary NLTK data
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('punkt', quiet=True)

In [ ]:
pos_tags = nltk.pos_tag(nltk.word_tokenize(sample_sentence))
print(f"POS Tags: {pos_tags}")

In [5]:
# Initialize NER pipeline
ner_pipeline = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"
)


Loading weights: 100%|███████████████████████| 199/199 [00:01<00:00, 110.16it/s, Materializing param=classifier.weight]
BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
snippet = text[:1000]
entities = ner_pipeline(snippet)

print(f"{'Entity':<20} | {'Type':<10} | {'Score':<5}")
print("-"*45)
for entity in entities:
    if entity['score'] > 0.90:
        print(f"{entity['word']:<20} | {entity['entity_group']:<10} | {entity['score']:.2f}")

NameError: name 'text' is not defined

In [ ]:
# Let's extract a specific section for summarization
transformer_section = """
The introduction of the Transformer architecture in the 2017 paper "Attention is all you need" was a watershed moment in AI. It provided a more effective and scalable way to handle sequential data like text, replacing older, less efficient methods like recurrence (RNNs) and convolutions.
The fundamental innovation of the Transformer is the attention mechanism. This component allows the model to weigh the importance of different words (tokens) in the input sequence when making a prediction. In essence, for each word it processes, the model can "pay attention" to all other words in the input, helping it understand context, resolve ambiguity, and handle long-range dependencies. This is crucial for tasks like translation, summarization, and question answering.
The Transformer architecture consists of an encoder stack (to process the input) and a decoder stack (to generate the output), both of which heavily utilize multi-head attention and feed-forward networks.
"""

In [ ]:
fast_sum = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")
res_fast = fast_sum(transformer_section, max_length=60, min_length=30, do_sample=False)
print(res_fast[0]['summary_text'])

In [ ]:
smart_sum = pipeline("summarization", model="facebook/bart-large-cnn")
res_smart = smart_sum(transformer_section, max_length=60, min_length=30, do_sample=False)
print(res_smart[0]['summary_text'])

In [ ]:
qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

In [ ]:
questions = [
    "What is the fundamental innovation of the Transformer?",
    "What are the risks of using Generative AI?"
]

for q in questions:
    res = qa_pipeline(question=q, context=text[:5000])
    print(f"\nQ: {q}")
    print(f"A: {res['answer']}")

In [ ]:
mask_filler = pipeline("fill-mask", model="bert-base-uncased")

In [ ]:
masked_sentence = "The goal of Generative AI is to create new [MASK]."
preds = mask_filler(masked_sentence)

for p in preds:
    print(f"{p['token_str']}: {p['score']:.2f}")